<!-- torchleet:colab -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fengyee/TorchLeet/blob/main/torch/easy/cuda-amp/cuda-amp.ipynb)

Check your work: `pip install -e python/` once from the repo root (on Colab: `!pip install -q "git+https://github.com/Fengyee/TorchLeet#subdirectory=python"`), then `from torchleet import check; check("cuda-amp", ...)`


## Problem: Implement Mixed Precision Training Using `torch.amp`

### Problem Statement
Mixed precision training runs most of the forward and backward pass in a 16-bit floating-point type (`float16` or `bfloat16`) while keeping the weights and numerically sensitive operations in `float32`. On GPUs with tensor cores this is faster and roughly halves activation memory, usually without hurting accuracy. Your task is to implement it with PyTorch's automatic mixed precision (AMP): `torch.autocast` and `torch.amp.GradScaler`.

> `torch.cuda.amp.autocast()` and `torch.cuda.amp.GradScaler()` are deprecated aliases. Use `torch.autocast("cuda", ...)` and `torch.amp.GradScaler("cuda")` instead.

### Background
- **What autocast does.** It picks a dtype per operation. Matmuls, convolutions and `nn.Linear` run in the low-precision dtype. Numerically sensitive ops run in `float32`: on CUDA that includes `sum`, softmax, `layer_norm`/`group_norm` and losses such as `mse_loss`; CPU autocast's `float32` list is shorter (mostly losses, `mse_loss` among them). The parameters themselves stay `float32` (the "master weights"), so small optimizer updates are not lost to rounding. Wrap only the forward pass and the loss: backward ops automatically run in the same dtype as their forward ops.
- **float16 vs bfloat16.**
  - `float16` has 5 exponent bits and 10 mantissa bits. Its largest value is 65504, its smallest normal value is about 6.1e-5, and its subnormals go down to about 6e-8. Small gradients underflow to zero, so float16 needs **loss scaling**.
  - `bfloat16` has 8 exponent bits, giving it the same range as `float32`, and 7 mantissa bits. Its precision is coarser, but underflow is not a problem, so it needs **no** `GradScaler`. Fast bfloat16 needs hardware support (NVIDIA Ampere or newer; CPUs with AVX-512 BF16 or AMX); elsewhere it is slow or, on older GPUs, unavailable.
- **GradScaler** (for float16 only):
  - `scaler.scale(loss)` multiplies the loss by a factor S (initially 2¹⁶), so the gradients computed by `backward()` stay representable in float16.
  - `scaler.step(optimizer)` divides the gradients by S. If any gradient is inf or NaN, it skips the update.
  - `scaler.update()` halves S after an overflow and doubles it after 2000 consecutive clean steps.
  - If you clip gradients, call `scaler.unscale_(optimizer)` first, so the clipping threshold applies to the true gradients rather than the scaled ones.

### Requirements
1. **`amp_config(device_type)`** → `(amp_dtype, scaler)`: `torch.float16` with an enabled `torch.amp.GradScaler` for `"cuda"`, and `torch.bfloat16` with a GradScaler built with `enabled=False` for `"cpu"`. A disabled scaler passes everything through unchanged, so the training step is the same code on both paths.
2. **`train_step(model, criterion, optimizer, scaler, inputs, labels, amp_dtype)`** → the batch's loss:
   - zero the gradients;
   - run the forward pass and the loss under `torch.autocast(device_type=inputs.device.type, dtype=amp_dtype)`;
   - backward through the scaler: `scaler.scale(loss).backward()`;
   - `scaler.unscale_(optimizer)`, then clip with `torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)`;
   - `scaler.step(optimizer)`, then `scaler.update()`.
3. Run inference under `torch.autocast` with no scaler, since there are no gradients to scale (provided).

### Grading
`check("cuda-amp", amp_config=amp_config, train_step=train_step)` needs no GPU. GradScaler's logic is the same on every device, so the grader drives `train_step` with CPU tensors, in `bfloat16` with a disabled scaler and in `float16` with an enabled CPU `GradScaler` (PyTorch ≥ 2.3), and compares the weights with a reference step. It includes a batch whose `float16` gradients underflow unless the loss is scaled, and one that overflows `float16`: that step must be skipped and the scale halved.

In [ ]:
# Implement mixed precision training in PyTorch using torch.amp
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

In [ ]:
# Define a simple model
class SimpleModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(10, 1)

    def forward(self, x):
        return self.fc(x)


device = "cuda" if torch.cuda.is_available() else "cpu"

# Generate synthetic data with a learnable (linear + noise) target
torch.manual_seed(0)
X = torch.randn(1000, 10)
y = X @ torch.randn(10, 1) + 0.1 * torch.randn(1000, 1)
dataset = TensorDataset(X, y)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

# Initialize model, loss function, and optimizer
model = SimpleModel().to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

In [ ]:
def amp_config(device_type):
    """
    Returns (amp_dtype, scaler) for device_type "cuda" or "cpu":
    float16 + an enabled torch.amp.GradScaler on CUDA, bfloat16 + a disabled one on CPU.
    """
    # TODO: float16 on CUDA, bfloat16 on CPU
    # TODO: a torch.amp.GradScaler that is enabled only when amp_dtype is float16
    ...


def train_step(model, criterion, optimizer, scaler, inputs, labels, amp_dtype):
    """One mixed-precision training step; returns the batch loss."""
    # TODO: zero the gradients
    # TODO: forward pass and loss under torch.autocast(device_type=inputs.device.type, dtype=amp_dtype)
    # TODO: scaled backward; unscale, then clip gradients to max_norm=1.0;
    #       optimizer step through the scaler; update the scale
    ...

In [ ]:
amp_dtype, scaler = amp_config(device)

# Checks: what autocast changes, and what it leaves alone
x = torch.randn(4, 10, device=device)
with torch.autocast(device_type=device, dtype=amp_dtype):
    out = model(x)
    check_loss = criterion(out, torch.zeros(4, 1, device=device))
assert out.dtype == amp_dtype, f"nn.Linear should run in {amp_dtype} under autocast, got {out.dtype}"
assert check_loss.dtype == torch.float32, "mse_loss is autocast to float32"
assert model.fc.weight.dtype == torch.float32, "the parameters (master weights) stay float32"
assert scaler.is_enabled() == (amp_dtype == torch.float16), "loss scaling is only needed for float16"
print("All checks passed")

In [ ]:
# Training loop
epochs = 5
for epoch in range(epochs):
    for inputs, labels in dataloader:
        inputs, labels = inputs.to(device), labels.to(device)
        loss = train_step(model, criterion, optimizer, scaler, inputs, labels, amp_dtype)

    print(f"Epoch {epoch + 1}/{epochs}, Loss: {float(loss):.4f}")

In [ ]:
# Inference: autocast without a scaler, since there are no gradients
model.eval()
X_test = torch.randn(5, 10, device=device)
with torch.inference_mode(), torch.autocast(device_type=device, dtype=amp_dtype):
    predictions = model(X_test)
print("Predictions:", predictions.float().squeeze(1), f"(computed in {predictions.dtype})")

In [ ]:
# Auto-grade (one-time setup from the repo root: pip install -e python/)
from torchleet import check
check("cuda-amp", amp_config=amp_config, train_step=train_step)